In [71]:
from time import sleep, time
import requests
import os
import pandas as pd
import json

api_key = '97D5425D-C330-49D5-A048-11D7B089DD91'

file = open('../data/OpenAPI/wlobs.json', 'r', encoding='utf8')
wlobs = json.load(file)
file.close()
for item in wlobs['content']:
    if not (item['addr'].startswith('서울특별시') or item['addr'].startswith('경기도')):
        continue
    wlobscd = item['wlobscd']
    site = f'../data/OpenAPI/wl_fw/{wlobscd}'
    if not os.path.isdir(site):
        os.mkdir(site)
    
    for yr in range(2012, 2022+1):
        for mo in range(1, 12+1):
            tic = time()
            
            if yr == 2022 and mo == 8:
                break
            if mo in [1, 3, 5, 7, 8, 10, 12]:
                dd = 31
            elif mo in [4, 6, 9, 11]:
                dd = 30
            elif mo == 2 and yr%4 == 0:
                dd = 29
            else:
                dd = 28
            url = f'http://api.hrfco.go.kr/{api_key}/waterlevel/list/10M/{wlobscd}/{yr}{mo:0>2}010000/{yr}{mo:0>2}{dd}2350.json'
            response = requests.get(url)
            response_json = response.json()
            if not (200 <= response.status_code < 300):
                print(response.status_code, wlobscd, yr, mo)
            df = pd.DataFrame(response_json['content'])[['ymdhm', 'wl', 'fw']]
            df = df.rename(columns={'wl': f'wl_{wlobscd}', 'fw': f'fw_{wlobscd}'})
            df['ymdhm'] = pd.to_datetime(df['ymdhm'])
            df.to_csv(f'{site}/wl_fw_{yr}{mo:0>2}.csv', index=False)
            
            toc = time()
            
            sleep( max(0.1 - (toc - tic), 0) )

In [61]:
def edit_ymdhm(ymdhm):
    yyyy, mm, dd, hh, mm = ymdhm[0:4], ymdhm[4:6], ymdhm[6:8], ymdhm[8:10], ymdhm[10:12]
    return f'{yyyy}-{mm}-{dd} {hh}:{mm}'